<a href="https://colab.research.google.com/github/Al-Amin95/PromptInjectionDetectionSystem/blob/data-analysis/notebooks/02_dataset_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Dataset preparation for model fine-tuning

#Part-01: Environment Setup for Dataset-Preparation

In [ ]:
# DATASET PREPARATION NOTEBOOK SETUP

from pathlib import Path

# CONFIG
REPO_URL = "https://github.com/Al-Amin95/PromptInjectionDetectionSystem.git"
BRANCH = "data-analysis"
REPO_DIR = Path("/content/PromptInjectionDetectionSystem")

# CLONE OR UPDATE REPO
if REPO_DIR.exists():
    print("Repo exists → updating...")
    %cd /content/PromptInjectionDetectionSystem
    !git checkout {BRANCH}
    !git pull
else:
    print("Cloning repo...")
    %cd /content
    !git clone -b {BRANCH} {REPO_URL}
    %cd /content/PromptInjectionDetectionSystem

# INSTALL REQUIREMENTS
print("Installing dependencies...")
!pip install -q -r requirements.txt

# DEFINE PATHS
PROJECT_ROOT = Path("/content/PromptInjectionDetectionSystem")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

# Dataset files
TRAIN_PATH = RAW_DATA_DIR / "train-00000-of-00001-9564e8b05b4757ab.parquet"
TEST_PATH  = RAW_DATA_DIR / "test-00000-of-00001-701d16158af87368.parquet"

# VERIFY FILES
print("\nChecking dataset files...")
print("Train exists:", TRAIN_PATH.exists())
print("Test exists :", TEST_PATH.exists())

# QUICK LOAD CHECK
import pandas as pd

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

print("\nDataset loaded successfully.")
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nColumns:", train_df.columns.tolist())

# Part-02: Dataset Loading & Validation

In [ ]:
# 1. Confirm train and test dataframes exist
required_dataframes = ["train_df", "test_df"]

for dataframe_name in required_dataframes:
    if dataframe_name not in globals():
        raise NameError(f"{dataframe_name} is missing. Run Part-01 setup first.")

print("Required dataframes found.")
print("train_df type:", type(train_df))
print("test_df type :", type(test_df))


# 2. Verify train and test dataset shapes
print("\nTrain shape:", train_df.shape)
print("Test shape :", test_df.shape)

assert train_df.shape == (546, 2), "Unexpected train_df shape."
assert test_df.shape == (116, 2), "Unexpected test_df shape."

print("Train and test dataset shapes are correct.")


# 3. Verify column structure
expected_columns = ["text", "label"]

print("\nTrain columns:", train_df.columns.tolist())
print("Test columns :", test_df.columns.tolist())

assert train_df.columns.tolist() == expected_columns, "Train columns are incorrect."
assert test_df.columns.tolist() == expected_columns, "Test columns are incorrect."

print("Column structure is correct.")


# 4. Verify label values and define label meaning
label_mapping = {
    0: "SAFE",
    1: "INJECTION"
}

train_unique_labels = sorted(train_df["label"].unique().tolist())
test_unique_labels = sorted(test_df["label"].unique().tolist())

print("\nTrain unique labels:", train_unique_labels)
print("Test unique labels :", test_unique_labels)
print("Label mapping:", label_mapping)

assert train_unique_labels == [0, 1], "Train labels are not valid binary labels."
assert test_unique_labels == [0, 1], "Test labels are not valid binary labels."

print("Label values and label meaning are correct.")


# 5. Preview raw samples
print("\nFirst 5 rows from train_df:")
display(train_df.head())

print("\nFirst 5 rows from test_df:")
display(test_df.head())


# Part-03: Create Output Folders

In [ ]:
# Define output folders
TABLES_DIR = RESULTS_DIR / "tables"
REPORTS_DIR = RESULTS_DIR / "reports"

# Create required folders
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Confirm folders
print("Output folders created or already exist.\n")

print("Processed data folder:", PROCESSED_DATA_DIR)
print("Tables folder        :", TABLES_DIR)
print("Reports folder       :", REPORTS_DIR)

print("\nFolder existence check:")
print("Processed data folder exists:", PROCESSED_DATA_DIR.exists())
print("Tables folder exists        :", TABLES_DIR.exists())
print("Reports folder exists       :", REPORTS_DIR.exists())

# Assertions
assert PROCESSED_DATA_DIR.exists(), "Processed data folder was not created."
assert TABLES_DIR.exists(), "Tables folder was not created."
assert REPORTS_DIR.exists(), "Reports folder was not created."


# Part-04: Preserve Original Prompt Text

In [ ]:
# Create copies before adding preparation columns
train_df = train_df.copy()
test_df = test_df.copy()

# Preserve original raw prompt text
train_df["original_text"] = train_df["text"]
test_df["original_text"] = test_df["text"]

# Confirm original_text column was created
print("Original prompt text preserved.\n")

print("Train columns:", train_df.columns.tolist())
print("Test columns :", test_df.columns.tolist())

print("\nTrain shape after adding original_text:", train_df.shape)
print("Test shape after adding original_text :", test_df.shape)

# Check that original_text exactly matches raw text
train_original_match = (train_df["text"] == train_df["original_text"]).all()
test_original_match = (test_df["text"] == test_df["original_text"]).all()

print("\nTrain original_text matches text:", train_original_match)
print("Test original_text matches text :", test_original_match)

# Assertions
assert "original_text" in train_df.columns, "original_text column missing in train_df."
assert "original_text" in test_df.columns, "original_text column missing in test_df."
assert train_original_match, "Train original_text does not match text."
assert test_original_match, "Test original_text does not match text."

# Part-05: Create Model Input Text Column

In [ ]:
# Define minimal text preparation function
def prepare_text_for_model(text):
    """
    Minimal preparation only.
    Preserve original prompt structure.
    Do not lowercase, remove punctuation, remove symbols,
    remove newlines, remove repeated spaces, or remove unusual characters.
    """
    return str(text)

# Create model input text column
train_df["text_for_model"] = train_df["original_text"].apply(prepare_text_for_model)
test_df["text_for_model"] = test_df["original_text"].apply(prepare_text_for_model)

# Confirm text_for_model column was created
print("Model input text column created.\n")

print("Train columns:", train_df.columns.tolist())
print("Test columns :", test_df.columns.tolist())

print("\nTrain shape after adding text_for_model:", train_df.shape)
print("Test shape after adding text_for_model :", test_df.shape)

# Check that text_for_model matches original_text
train_text_for_model_match = (train_df["original_text"] == train_df["text_for_model"]).all()
test_text_for_model_match = (test_df["original_text"] == test_df["text_for_model"]).all()

print("\nTrain text_for_model matches original_text:", train_text_for_model_match)
print("Test text_for_model matches original_text :", test_text_for_model_match)

# Assertions
assert "text_for_model" in train_df.columns, "text_for_model column missing in train_df."
assert "text_for_model" in test_df.columns, "text_for_model column missing in test_df."
assert train_text_for_model_match, "Train text_for_model does not match original_text."
assert test_text_for_model_match, "Test text_for_model does not match original_text."


# Part-06: Text Preparation Impact Summary

In [ ]:
# Common save helper for GitHub repo folder and Google Drive

from pathlib import Path
import shutil
import pandas as pd

# Google Drive project folders
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/USW_Dissertation/Prompt-Injection-Detection-System-SHAP")
DRIVE_TABLES_DIR = DRIVE_PROJECT_DIR / "results" / "tables"

# Ensure folders exist
TABLES_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_TABLES_DIR.mkdir(parents=True, exist_ok=True)

def save_table_to_github_and_drive(dataframe, file_name):
    """
    Save a summary table to the GitHub repo folder and copy it to Google Drive.
    """
    github_path = TABLES_DIR / file_name
    drive_path = DRIVE_TABLES_DIR / file_name

    dataframe.to_csv(github_path, index=False)
    shutil.copy2(github_path, drive_path)

    print(f"Saved to GitHub repo: {github_path}")
    print(f"Saved to Google Drive: {drive_path}")

    assert github_path.exists(), f"{file_name} was not saved in GitHub repo."
    assert drive_path.exists(), f"{file_name} was not saved in Google Drive."
    assert github_path.stat().st_size > 0, f"{file_name} in GitHub repo is empty."
    assert drive_path.stat().st_size > 0, f"{file_name} in Google Drive is empty."

    return github_path, drive_path

print("GitHub and Google Drive table-saving helper is ready.")

In [ ]:

# Count changed rows after creating text_for_model
train_changed_count = (train_df["original_text"] != train_df["text_for_model"]).sum()
test_changed_count = (test_df["original_text"] != test_df["text_for_model"]).sum()

# Calculate percentages
train_changed_percentage = round((train_changed_count / len(train_df)) * 100, 2)
test_changed_percentage = round((test_changed_count / len(test_df)) * 100, 2)

# Create preparation impact summary
text_preparation_impact_summary = pd.DataFrame([
    {
        "split": "train",
        "total_rows": len(train_df),
        "changed_rows": train_changed_count,
        "changed_percentage": train_changed_percentage
    },
    {
        "split": "test",
        "total_rows": len(test_df),
        "changed_rows": test_changed_count,
        "changed_percentage": test_changed_percentage
    }
])

print("Text preparation impact summary:")
display(text_preparation_impact_summary)

# Save to GitHub repo folder and Google Drive
final_cleaning_github_path, final_cleaning_drive_path = save_table_to_github_and_drive(
    text_preparation_impact_summary,
    "final_cleaning_summary.csv"
)

# Assertions
assert train_changed_count == 0, "Some train text values changed during preparation."
assert test_changed_count == 0, "Some test text values changed during preparation."


# Part-07: Keep Official Test Set Untouched

In [ ]:
# Create a protected copy of the official test set
final_test_df = test_df.copy()

print("Official test set copied and protected.\n")

# Confirm test set shape and columns
print("Original test_df shape:", test_df.shape)
print("Final test_df shape   :", final_test_df.shape)

print("\nOriginal test_df columns:", test_df.columns.tolist())
print("Final test_df columns   :", final_test_df.columns.tolist())

# Confirm final_test_df is identical to test_df at this stage
test_set_unchanged = final_test_df.equals(test_df)

print("\nFinal test set unchanged:", test_set_unchanged)

# Confirm row count remains official test size
print("Final test set row count:", len(final_test_df))

# Assertions
assert final_test_df.shape == test_df.shape, "Final test set shape changed."
assert final_test_df.columns.tolist() == test_df.columns.tolist(), "Final test set columns changed."
assert test_set_unchanged, "Final test set does not match original test_df."
assert len(final_test_df) == 116, "Official test set row count should remain 116."



# Part-08: Create Validation Split from Training Data

In [ ]:
from sklearn.model_selection import train_test_split

# Create validation split only from the original training dataset
final_train_df, validation_df = train_test_split(
    train_df,
    test_size=0.20,
    stratify=train_df["label"],
    random_state=42
)

# Create copies after splitting
final_train_df = final_train_df.copy()
validation_df = validation_df.copy()

print("Validation split created from training data only.\n")

# Print split shapes
print("Original train_df shape:", train_df.shape)
print("Final train_df shape   :", final_train_df.shape)
print("Validation_df shape    :", validation_df.shape)
print("Final test_df shape    :", final_test_df.shape)

# Print label distributions
print("\nOriginal train label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nFinal train label distribution:")
print(final_train_df["label"].value_counts().sort_index())

print("\nValidation label distribution:")
print(validation_df["label"].value_counts().sort_index())

print("\nFinal test label distribution:")
print(final_test_df["label"].value_counts().sort_index())

# Confirm final split sizes
assert final_train_df.shape[0] == 436, "Final train size should be 436."
assert validation_df.shape[0] == 110, "Validation size should be 110."
assert final_test_df.shape[0] == 116, "Final test size should remain 116."

# Confirm columns remain unchanged after split
assert final_train_df.columns.tolist() == train_df.columns.tolist(), "Final train columns changed."
assert validation_df.columns.tolist() == train_df.columns.tolist(), "Validation columns changed."

# Confirm test set was not used for validation
assert final_test_df.equals(test_df), "Final test set was changed."


# Part-09: Add Split Column

In [ ]:
# Add split column to each final dataset
final_train_df["split"] = "train"
validation_df["split"] = "validation"
final_test_df["split"] = "test"

print("Split column added to final datasets.\n")

# Confirm columns
print("Final train columns:", final_train_df.columns.tolist())
print("Validation columns :", validation_df.columns.tolist())
print("Final test columns :", final_test_df.columns.tolist())

# Confirm shapes
print("\nFinal train shape:", final_train_df.shape)
print("Validation shape :", validation_df.shape)
print("Final test shape :", final_test_df.shape)

# Confirm split values
print("\nFinal train split values:")
print(final_train_df["split"].value_counts())

print("\nValidation split values:")
print(validation_df["split"].value_counts())

print("\nFinal test split values:")
print(final_test_df["split"].value_counts())

# Assertions
assert "split" in final_train_df.columns, "split column missing in final_train_df."
assert "split" in validation_df.columns, "split column missing in validation_df."
assert "split" in final_test_df.columns, "split column missing in final_test_df."

assert final_train_df["split"].eq("train").all(), "Incorrect split value in final_train_df."
assert validation_df["split"].eq("validation").all(), "Incorrect split value in validation_df."
assert final_test_df["split"].eq("test").all(), "Incorrect split value in final_test_df."


# Part-10: Add Stable Row IDs

In [ ]:
# Reset indexes before assigning stable IDs
final_train_df = final_train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
final_test_df = final_test_df.reset_index(drop=True)

# Add stable row IDs
final_train_df["id"] = [f"train_{i:06d}" for i in range(1, len(final_train_df) + 1)]
validation_df["id"] = [f"validation_{i:06d}" for i in range(1, len(validation_df) + 1)]
final_test_df["id"] = [f"test_{i:06d}" for i in range(1, len(final_test_df) + 1)]

print("Stable row IDs added.\n")

# Confirm columns
print("Final train columns:", final_train_df.columns.tolist())
print("Validation columns :", validation_df.columns.tolist())
print("Final test columns :", final_test_df.columns.tolist())

# Confirm ID examples
print("\nFirst 5 train IDs:")
print(final_train_df["id"].head().tolist())

print("\nFirst 5 validation IDs:")
print(validation_df["id"].head().tolist())

print("\nFirst 5 test IDs:")
print(final_test_df["id"].head().tolist())

# Confirm ID uniqueness
train_id_unique = final_train_df["id"].is_unique
validation_id_unique = validation_df["id"].is_unique
test_id_unique = final_test_df["id"].is_unique

print("\nTrain IDs unique:", train_id_unique)
print("Validation IDs unique:", validation_id_unique)
print("Test IDs unique:", test_id_unique)

# Confirm ID counts
print("\nTrain ID count:", final_train_df["id"].nunique())
print("Validation ID count:", validation_df["id"].nunique())
print("Test ID count:", final_test_df["id"].nunique())

# Assertions
assert "id" in final_train_df.columns, "id column missing in final_train_df."
assert "id" in validation_df.columns, "id column missing in validation_df."
assert "id" in final_test_df.columns, "id column missing in final_test_df."

assert train_id_unique, "Train IDs are not unique."
assert validation_id_unique, "Validation IDs are not unique."
assert test_id_unique, "Test IDs are not unique."

assert final_train_df["id"].str.startswith("train_").all(), "Train ID prefix is incorrect."
assert validation_df["id"].str.startswith("validation_").all(), "Validation ID prefix is incorrect."
assert final_test_df["id"].str.startswith("test_").all(), "Test ID prefix is incorrect."


# Part-11: Class Distribution Analysis

In [ ]:
# Define label mapping
label_mapping = {
    0: "SAFE",
    1: "INJECTION"
}

# Function to calculate class distribution
def calculate_class_distribution(dataframe, split_name):
    distribution = (
        dataframe["label"]
        .value_counts()
        .sort_index()
        .reset_index()
    )

    distribution.columns = ["label", "count"]
    distribution["label_name"] = distribution["label"].map(label_mapping)
    distribution["split"] = split_name
    distribution["total_rows"] = len(dataframe)
    distribution["percentage"] = round((distribution["count"] / len(dataframe)) * 100, 2)

    return distribution[
        ["split", "label", "label_name", "count", "total_rows", "percentage"]
    ]

# Calculate distribution for each final split
final_train_distribution = calculate_class_distribution(final_train_df, "train")
validation_distribution = calculate_class_distribution(validation_df, "validation")
final_test_distribution = calculate_class_distribution(final_test_df, "test")

# Combine distributions
final_split_distribution_summary = pd.concat(
    [
        final_train_distribution,
        validation_distribution,
        final_test_distribution
    ],
    ignore_index=True
)

print("Final split class distribution summary:")
display(final_split_distribution_summary)

# Save to GitHub repo folder and Google Drive
final_split_distribution_github_path, final_split_distribution_drive_path = save_table_to_github_and_drive(
    final_split_distribution_summary,
    "final_split_distribution_summary.csv"
)

# Assertions
assert final_train_df.shape[0] == 436, "Final train row count is incorrect."
assert validation_df.shape[0] == 110, "Validation row count is incorrect."
assert final_test_df.shape[0] == 116, "Final test row count is incorrect."

assert final_train_df["label"].value_counts().sort_index().to_dict() == {0: 274, 1: 162}, "Final train label distribution is incorrect."
assert validation_df["label"].value_counts().sort_index().to_dict() == {0: 69, 1: 41}, "Validation label distribution is incorrect."
assert final_test_df["label"].value_counts().sort_index().to_dict() == {0: 56, 1: 60}, "Final test label distribution is incorrect."


# Part-12: Final Prepared Data Quality Check

In [ ]:
# Function to check final prepared data quality
def final_quality_check(dataframe, split_name):
    return {
        "split": split_name,
        "total_rows": len(dataframe),
        "missing_text_for_model": dataframe["text_for_model"].isna().sum(),
        "empty_text_for_model": dataframe["text_for_model"].astype(str).eq("").sum(),
        "whitespace_only_text_for_model": dataframe["text_for_model"].astype(str).str.strip().eq("").sum(),
        "missing_label": dataframe["label"].isna().sum(),
        "invalid_label": (~dataframe["label"].isin([0, 1])).sum(),
        "missing_id": dataframe["id"].isna().sum(),
        "missing_split": dataframe["split"].isna().sum()
    }

# Create quality summary
final_quality_summary = pd.DataFrame([
    final_quality_check(final_train_df, "train"),
    final_quality_check(validation_df, "validation"),
    final_quality_check(final_test_df, "test")
])

print("Final prepared data quality summary:")
display(final_quality_summary)

# Save to GitHub repo folder and Google Drive
final_quality_github_path, final_quality_drive_path = save_table_to_github_and_drive(
    final_quality_summary,
    "final_quality_summary.csv"
)

# Assertions
assert final_quality_summary["missing_text_for_model"].sum() == 0, "Missing text_for_model values found."
assert final_quality_summary["empty_text_for_model"].sum() == 0, "Empty text_for_model values found."
assert final_quality_summary["whitespace_only_text_for_model"].sum() == 0, "Whitespace-only text_for_model values found."
assert final_quality_summary["missing_label"].sum() == 0, "Missing label values found."
assert final_quality_summary["invalid_label"].sum() == 0, "Invalid label values found."
assert final_quality_summary["missing_id"].sum() == 0, "Missing ID values found."
assert final_quality_summary["missing_split"].sum() == 0, "Missing split values found."


# Part-13: Final Duplicate & Leakage Check

In [ ]:
# Check duplicate text_for_model values within each final split
train_duplicate_text_count = final_train_df["text_for_model"].duplicated().sum()
validation_duplicate_text_count = validation_df["text_for_model"].duplicated().sum()
test_duplicate_text_count = final_test_df["text_for_model"].duplicated().sum()

# Create text sets for overlap checks
train_text_set = set(final_train_df["text_for_model"])
validation_text_set = set(validation_df["text_for_model"])
test_text_set = set(final_test_df["text_for_model"])

# Check overlap between final splits
train_validation_overlap_count = len(train_text_set.intersection(validation_text_set))
train_test_overlap_count = len(train_text_set.intersection(test_text_set))
validation_test_overlap_count = len(validation_text_set.intersection(test_text_set))

# Check conflicting labels across all prepared data
combined_final_df = pd.concat(
    [
        final_train_df,
        validation_df,
        final_test_df
    ],
    ignore_index=True
)

label_counts_per_text = (
    combined_final_df
    .groupby("text_for_model")["label"]
    .nunique()
    .reset_index(name="unique_label_count")
)

conflicting_label_count = (label_counts_per_text["unique_label_count"] > 1).sum()

# Create duplicate and leakage summary
final_duplicate_leakage_summary = pd.DataFrame([
    {
        "check_name": "train_duplicate_text_for_model",
        "issue_count": train_duplicate_text_count,
        "status": "Pass" if train_duplicate_text_count == 0 else "Review"
    },
    {
        "check_name": "validation_duplicate_text_for_model",
        "issue_count": validation_duplicate_text_count,
        "status": "Pass" if validation_duplicate_text_count == 0 else "Review"
    },
    {
        "check_name": "test_duplicate_text_for_model",
        "issue_count": test_duplicate_text_count,
        "status": "Pass" if test_duplicate_text_count == 0 else "Review"
    },
    {
        "check_name": "train_validation_overlap",
        "issue_count": train_validation_overlap_count,
        "status": "Pass" if train_validation_overlap_count == 0 else "Review"
    },
    {
        "check_name": "train_test_overlap",
        "issue_count": train_test_overlap_count,
        "status": "Pass" if train_test_overlap_count == 0 else "Review"
    },
    {
        "check_name": "validation_test_overlap",
        "issue_count": validation_test_overlap_count,
        "status": "Pass" if validation_test_overlap_count == 0 else "Review"
    },
    {
        "check_name": "conflicting_labels_after_preparation",
        "issue_count": conflicting_label_count,
        "status": "Pass" if conflicting_label_count == 0 else "Review"
    }
])

print("Final duplicate and leakage summary:")
display(final_duplicate_leakage_summary)

# Save to GitHub repo folder and Google Drive
final_duplicate_leakage_github_path, final_duplicate_leakage_drive_path = save_table_to_github_and_drive(
    final_duplicate_leakage_summary,
    "final_duplicate_leakage_summary.csv"
)

# Assertions
assert train_duplicate_text_count == 0, "Duplicate text found in final train split."
assert validation_duplicate_text_count == 0, "Duplicate text found in validation split."
assert test_duplicate_text_count == 0, "Duplicate text found in final test split."

assert train_validation_overlap_count == 0, "Train-validation overlap found."
assert train_test_overlap_count == 0, "Train-test overlap found."
assert validation_test_overlap_count == 0, "Validation-test overlap found."

assert conflicting_label_count == 0, "Conflicting labels found after preparation."



# Part-14: Select Final Model-Ready Columns


In [ ]:
# Define final required columns for model preparation
FINAL_COLUMNS = [
    "id",
    "original_text",
    "text_for_model",
    "label",
    "split"
]

# Select only final model-ready columns
final_train_df = final_train_df[FINAL_COLUMNS].copy()
validation_df = validation_df[FINAL_COLUMNS].copy()
final_test_df = final_test_df[FINAL_COLUMNS].copy()

print("Final model-ready columns selected.\n")

# Confirm final columns
print("Final train columns:", final_train_df.columns.tolist())
print("Validation columns :", validation_df.columns.tolist())
print("Final test columns :", final_test_df.columns.tolist())

# Confirm final shapes
print("\nFinal train shape:", final_train_df.shape)
print("Validation shape :", validation_df.shape)
print("Final test shape :", final_test_df.shape)

# Preview final datasets
print("\nFinal train preview:")
display(final_train_df.head())

print("\nValidation preview:")
display(validation_df.head())

print("\nFinal test preview:")
display(final_test_df.head())

# Assertions
assert final_train_df.columns.tolist() == FINAL_COLUMNS, "Final train columns are incorrect."
assert validation_df.columns.tolist() == FINAL_COLUMNS, "Validation columns are incorrect."
assert final_test_df.columns.tolist() == FINAL_COLUMNS, "Final test columns are incorrect."

assert final_train_df.shape == (436, 5), "Final train shape is incorrect."
assert validation_df.shape == (110, 5), "Validation shape is incorrect."
assert final_test_df.shape == (116, 5), "Final test shape is incorrect."


# Part-15: Final Schema Check

In [ ]:
# Required final schema
FINAL_COLUMNS = [
    "id",
    "original_text",
    "text_for_model",
    "label",
    "split"
]

print("Running final schema check...\n")

# 1. Check columns
print("Final train columns:", final_train_df.columns.tolist())
print("Validation columns :", validation_df.columns.tolist())
print("Final test columns :", final_test_df.columns.tolist())

# 2. Check row counts
print("\nFinal train row count:", len(final_train_df))
print("Validation row count :", len(validation_df))
print("Final test row count :", len(final_test_df))

# 3. Check labels
print("\nFinal train labels:", sorted(final_train_df["label"].unique().tolist()))
print("Validation labels :", sorted(validation_df["label"].unique().tolist()))
print("Final test labels :", sorted(final_test_df["label"].unique().tolist()))

# 4. Check missing model input text
print("\nMissing text_for_model values:")
print("Final train:", final_train_df["text_for_model"].isna().sum())
print("Validation :", validation_df["text_for_model"].isna().sum())
print("Final test :", final_test_df["text_for_model"].isna().sum())

# 5. Check ID uniqueness
print("\nID uniqueness:")
print("Final train IDs unique:", final_train_df["id"].is_unique)
print("Validation IDs unique :", validation_df["id"].is_unique)
print("Final test IDs unique :", final_test_df["id"].is_unique)

# 6. Check split values
print("\nSplit values:")
print("Final train:", final_train_df["split"].unique().tolist())
print("Validation :", validation_df["split"].unique().tolist())
print("Final test :", final_test_df["split"].unique().tolist())

# 7. Check overlap between final splits
train_text_set = set(final_train_df["text_for_model"])
validation_text_set = set(validation_df["text_for_model"])
test_text_set = set(final_test_df["text_for_model"])

train_validation_overlap = len(train_text_set.intersection(validation_text_set))
train_test_overlap = len(train_text_set.intersection(test_text_set))
validation_test_overlap = len(validation_text_set.intersection(test_text_set))

print("\nOverlap check:")
print("Train-validation overlap:", train_validation_overlap)
print("Train-test overlap      :", train_test_overlap)
print("Validation-test overlap :", validation_test_overlap)

# Assertions
assert final_train_df.columns.tolist() == FINAL_COLUMNS, "Final train schema is incorrect."
assert validation_df.columns.tolist() == FINAL_COLUMNS, "Validation schema is incorrect."
assert final_test_df.columns.tolist() == FINAL_COLUMNS, "Final test schema is incorrect."

assert len(final_train_df) == 436, "Final train row count is incorrect."
assert len(validation_df) == 110, "Validation row count is incorrect."
assert len(final_test_df) == 116, "Final test row count is incorrect."

assert final_train_df["label"].isin([0, 1]).all(), "Invalid labels found in final train."
assert validation_df["label"].isin([0, 1]).all(), "Invalid labels found in validation."
assert final_test_df["label"].isin([0, 1]).all(), "Invalid labels found in final test."

assert final_train_df["text_for_model"].isna().sum() == 0, "Missing text_for_model in final train."
assert validation_df["text_for_model"].isna().sum() == 0, "Missing text_for_model in validation."
assert final_test_df["text_for_model"].isna().sum() == 0, "Missing text_for_model in final test."

assert final_train_df["id"].is_unique, "Final train IDs are not unique."
assert validation_df["id"].is_unique, "Validation IDs are not unique."
assert final_test_df["id"].is_unique, "Final test IDs are not unique."

assert final_train_df["split"].eq("train").all(), "Incorrect split value in final train."
assert validation_df["split"].eq("validation").all(), "Incorrect split value in validation."
assert final_test_df["split"].eq("test").all(), "Incorrect split value in final test."

assert train_validation_overlap == 0, "Train-validation overlap found."
assert train_test_overlap == 0, "Train-test overlap found."
assert validation_test_overlap == 0, "Validation-test overlap found."


# Part-16: Save Processed Dataset Files

In [ ]:
# Part-16: Save Preparation Summary to GitHub Repo and Google Drive

from pathlib import Path
import shutil
import pandas as pd

# Define Google Drive folders
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/USW_Dissertation/Prompt-Injection-Detection-System-SHAP")
DRIVE_PROCESSED_DATA_DIR = DRIVE_PROJECT_DIR / "data" / "processed"
DRIVE_TABLES_DIR = DRIVE_PROJECT_DIR / "results" / "tables"

# Create Google Drive folders if needed
DRIVE_PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Required processed dataset files
processed_dataset_files = [
    PROCESSED_DATA_DIR / "clean_train.csv",
    PROCESSED_DATA_DIR / "clean_validation.csv",
    PROCESSED_DATA_DIR / "clean_test.csv",
    PROCESSED_DATA_DIR / "clean_train.parquet",
    PROCESSED_DATA_DIR / "clean_validation.parquet",
    PROCESSED_DATA_DIR / "clean_test.parquet"
]

# Required preparation summary files already created in earlier parts
existing_summary_files = [
    TABLES_DIR / "final_cleaning_summary.csv",
    TABLES_DIR / "final_split_distribution_summary.csv",
    TABLES_DIR / "final_quality_summary.csv",
    TABLES_DIR / "final_duplicate_leakage_summary.csv"
]

# Copy existing summary files from GitHub repo folder to Google Drive
for summary_file in existing_summary_files:
    assert summary_file.exists(), f"Missing GitHub repo summary file: {summary_file}"
    drive_summary_file = DRIVE_TABLES_DIR / summary_file.name
    shutil.copy2(summary_file, drive_summary_file)

# Create final processed files summary
summary_rows = []

# Add processed dataset files to summary
for file_path in processed_dataset_files:
    drive_file_path = DRIVE_PROCESSED_DATA_DIR / file_path.name

    summary_rows.append({
        "category": "processed_dataset",
        "file_name": file_path.name,
        "github_repo_path": str(file_path),
        "github_repo_exists": file_path.exists(),
        "github_repo_size_bytes": file_path.stat().st_size if file_path.exists() else 0,
        "google_drive_path": str(drive_file_path),
        "google_drive_exists": drive_file_path.exists(),
        "google_drive_size_bytes": drive_file_path.stat().st_size if drive_file_path.exists() else 0
    })

# Add summary table files to summary
for file_path in existing_summary_files:
    drive_file_path = DRIVE_TABLES_DIR / file_path.name

    summary_rows.append({
        "category": "summary_table",
        "file_name": file_path.name,
        "github_repo_path": str(file_path),
        "github_repo_exists": file_path.exists(),
        "github_repo_size_bytes": file_path.stat().st_size if file_path.exists() else 0,
        "google_drive_path": str(drive_file_path),
        "google_drive_exists": drive_file_path.exists(),
        "google_drive_size_bytes": drive_file_path.stat().st_size if drive_file_path.exists() else 0
    })

final_processed_files_summary = pd.DataFrame(summary_rows)

print("Final processed files summary:")
display(final_processed_files_summary)

# Save final processed files summary to GitHub repo folder
final_processed_files_summary_github_path = TABLES_DIR / "final_processed_files_summary.csv"
final_processed_files_summary.to_csv(final_processed_files_summary_github_path, index=False)

# Copy final processed files summary to Google Drive
final_processed_files_summary_drive_path = DRIVE_TABLES_DIR / "final_processed_files_summary.csv"
shutil.copy2(final_processed_files_summary_github_path, final_processed_files_summary_drive_path)

print("\nFinal processed files summary saved to GitHub repo:")
print(final_processed_files_summary_github_path)

print("\nFinal processed files summary copied to Google Drive:")
print(final_processed_files_summary_drive_path)

# Confirm all files exist
print("\nGitHub repo file check:")
for file_path in processed_dataset_files + existing_summary_files + [final_processed_files_summary_github_path]:
    print(file_path, "exists:", file_path.exists())

print("\nGoogle Drive file check:")
drive_required_files = (
    [DRIVE_PROCESSED_DATA_DIR / file_path.name for file_path in processed_dataset_files]
    + [DRIVE_TABLES_DIR / file_path.name for file_path in existing_summary_files]
    + [final_processed_files_summary_drive_path]
)

for file_path in drive_required_files:
    print(file_path, "exists:", file_path.exists())

# Assertions
assert final_processed_files_summary["github_repo_exists"].all(), "Some GitHub repo files are missing."
assert final_processed_files_summary["google_drive_exists"].all(), "Some Google Drive files are missing."

assert (final_processed_files_summary["github_repo_size_bytes"] > 0).all(), "Some GitHub repo files are empty."
assert (final_processed_files_summary["google_drive_size_bytes"] > 0).all(), "Some Google Drive files are empty."

assert final_processed_files_summary_github_path.exists(), "final_processed_files_summary.csv missing in GitHub repo."
assert final_processed_files_summary_drive_path.exists(), "final_processed_files_summary.csv missing in Google Drive."
